# 02 · Train all models on FaceForensics++

Trains every model in `MODELS` with **the same recipe**: same FF++ training data, 224×224 input, augmentation
(flip, colour jitter, JPEG compression, random erasing), AdamW + OneCycle, 8 epochs with early stopping on
validation AUC, and a decision threshold tuned on FF++ validation.

| Model | Why it's in the comparison |
|---|---|
| `efficientnet_b0` | proposed model: small, fast, Grad-CAM++-friendly |
| `meso4` | MesoNet-4: small purpose-built deepfake CNN (Afchar et al. 2018), FF++ paper baseline, trained from scratch |
| `legacy_xception` | Xception: the FaceForensics++ paper's reference detector |
| `resnet50` | standard CNN baseline |
| `efficientnet_b4` | larger EfficientNet (DFDC-winning family) |
| `vit_small_patch16_224` | Vision Transformer (ViT-S/16) |
| `convnext_tiny` | ConvNeXt-T: modern CNN redesigned with transformer-era ideas (Liu et al. 2022) |

**Time:** roughly 7–8 GPU-hours for all seven on a T4 (EfficientNet-B0 alone ≈ 40 min). Kaggle stops a session at
12 h and **loses the output if it times out**, so this notebook skips models that won't fit in `TIME_BUDGET_H`.
Safer: make a **copy** of this notebook and split `MODELS` between the two (e.g. copy A = B0, Meso, ResNet, ViT;
copy B = Xception, B4, ConvNeXt). Notebook 03 picks up the models from all attached copies.

**Notebook settings (right sidebar)**
- **Accelerator:** GPU T4 x2 · **Internet:** On
- **Inputs:** output of **01_extract_faces** (FF++)

**Produces:** `/kaggle/working/outputs/checkpoints/` (EfficientNet-B0 in `best.pt`, every other model in `<model>/best.pt`).

In [ ]:
# --- Project code -------------------------------------------------------------------------
# Always clones the latest code. It lives in /tmp, OUTSIDE /kaggle/working, so it is never saved into this
# notebook's output (otherwise later notebooks would pick up a stale copy). For a private repo add a
# GITHUB_TOKEN secret. Fallback: a zip of the project uploaded as a Kaggle *dataset*.
import os, sys, glob, shutil, subprocess

PROJECT = "/tmp/dfdc-project"
REPO, BRANCH = "https://github.com/Amay-M-Nair/DFDC.git", "dev"

if not os.path.exists(PROJECT):
    token = ""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT], capture_output=True)
    if r.returncode == 0:
        subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO])  # don't keep the token
    else:
        found = [p for d in range(5) for p in glob.glob("/kaggle/input/" + "*/" * d + "src/config.py")]
        if not found:
            raise RuntimeError("git clone failed: add a GITHUB_TOKEN secret or attach the project zip as a dataset")
        print("WARNING: git clone failed; using the attached copy at", os.path.dirname(os.path.dirname(found[0])))
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), PROJECT,
                        ignore=shutil.ignore_patterns("data", "outputs", "__pycache__"))

shutil.rmtree("/kaggle/working/project", ignore_errors=True)  # remove stale copies saved by older notebook versions
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip install -q grad-cam anthropic
!git -C {PROJECT} log -1 --format="code version: %h  %s  (%cd)" --date=short 2>/dev/null || echo "code version: attached copy (no git)"

In [ ]:
from src.kaggle_utils import setup
setup(data=True)

import json, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown, Image as Show

import time, torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

## Settings

In [ ]:
MODELS = ["efficientnet_b0", "meso4", "resnet50", "vit_small_patch16_224", "legacy_xception", "efficientnet_b4",
          "convnext_tiny"]
EPOCHS, BATCH, PATIENCE, WEIGHT_DECAY, SUBSET = 8, 32, 3, 0.05, None
LR = {"default": 3e-4, "meso4": 1e-3, "vit_small_patch16_224": 1e-4, "convnext_tiny": 1e-4}   # from-scratch Meso needs more, ViT less
TIME_BUDGET_H = 11.0

# Rough training cost relative to EfficientNet-B0, used to skip models that would overrun the session
RELATIVE_COST = {"efficientnet_b0": 1.0, "meso4": 0.7, "resnet50": 1.3, "vit_small_patch16_224": 1.3,
                 "legacy_xception": 1.8, "efficientnet_b4": 2.5, "convnext_tiny": 1.5}

## Train

In [ ]:
start, unit_h, log = time.time(), None, []
for m in MODELS:
    elapsed_h = (time.time() - start) / 3600
    estimate_h = unit_h * RELATIVE_COST.get(m, 2.0) if unit_h else 0
    if elapsed_h + estimate_h > TIME_BUDGET_H:
        print(f"SKIP {m}: ~{estimate_h:.1f} h needed, {TIME_BUDGET_H - elapsed_h:.1f} h left. Train it in a copy of this notebook.")
        log.append({"model": m, "status": "skipped"}); continue
    print(f"\n{'=' * 30} {m} {'=' * 30}")
    t0 = time.time()
    args = (f"--backbone {m} --epochs {EPOCHS} --batch-size {BATCH} --lr {LR.get(m, LR['default'])} "
            f"--patience {PATIENCE} --weight-decay {WEIGHT_DECAY} --workers 4")
    if SUBSET: args += f" --subset {SUBSET}"
    !TQDM_MININTERVAL=60 python -m src.classifier.train {args}
    hours = (time.time() - t0) / 3600
    unit_h = unit_h or hours / RELATIVE_COST.get(m, 2.0)
    log.append({"model": m, "status": "done", "hours": round(hours, 2)})
display(pd.DataFrame(log))

## Training curves (all models)

In [ ]:
from pathlib import Path
ck = Path("/kaggle/working/outputs/checkpoints")
histories = {("efficientnet_b0" if h.parent == ck else h.parent.name): pd.read_json(h) for h in ck.rglob("history.json")}
fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4))
for m, h in histories.items():
    a.plot(h.epoch, h.train_loss, marker="o", label=m)
    b.plot(h.epoch, h.val_auc, marker="o", label=m)
a.set_title("train loss"); b.set_title("FF++ validation AUC")
for ax in (a, b): ax.set_xlabel("epoch"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

best = pd.DataFrame([{"model": m, "best_epoch": int(h.loc[h.val_auc.idxmax(), "epoch"]), "val_auc": h.val_auc.max(),
                      "val_acc_tuned": h.loc[h.val_auc.idxmax(), "val_acc_tuned"], "minutes": h.elapsed_min.iloc[-1]}
                     for m, h in histories.items()]).sort_values("val_auc", ascending=False)
display(best.round(4))